# Día 7 - Pipelines, forecasting económico y presentación de resultados

**Fecha:** 21-10-2026

Hoy cerramos el flujo completo:

```text
DATOS -> VALIDACIÓN -> PREPROCESAMIENTO -> MODELO -> MÉTRICAS -> INTERPRETACIÓN -> COMUNICACIÓN
```

## Objetivos

- automatizar preprocesamiento con Pipeline/ColumnTransformer;
- evitar leakage durante transformación;
- convertir una serie temporal en aprendizaje supervisado con lags;
- comparar contra un baseline temporal;
- entender por qué no se hace split aleatorio en forecasting;
- estructurar el proyecto final.

## 0. Librerías y carpeta de datos

No hay instalación automática. Si una librería opcional falta, el notebook mostrará el comando que puedes ejecutar manualmente en una celda nueva. Todos los imports principales quedan juntos para que sea claro qué herramientas usa la jornada.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, TimeSeriesSplit

In [ ]:
DATA_DIR = Path('../datos')
if not DATA_DIR.exists():
    DATA_DIR = Path('datos')
print('Datos:', DATA_DIR.resolve())

A partir de aquí, cada bloque de código irá acompañado de una explicación de **qué entra, qué sale y qué debemos mirar**. Ejecutar una celda sin interpretar el resultado no cuenta como comprender el modelo.

# 1. ¿Por qué usar Pipeline?

En ejemplos anteriores hicimos pasos manuales para ver qué ocurría. En un proyecto real necesitamos que el flujo sea reproducible.

Pipeline garantiza que:

- el imputador aprende solo de Train;
- el escalador aprende solo de Train;
- el one-hot aprende categorías desde Train;
- las mismas transformaciones se aplican a Validation/Test;
- el modelo recibe datos consistentes.

Esto reduce errores y leakage.

Hasta ahora hicimos muchos pasos a mano para entenderlos. En un proyecto real, separar limpieza, escalado, one-hot y modelo en celdas sueltas crea riesgo de:

- olvidar un paso al predecir nuevos datos;
- aplicar transformaciones distintas entre Train y producción;
- calcular estadísticas usando todo el dataset y generar leakage.

`Pipeline` encapsula una **secuencia reproducible**. Durante `fit()`, cada transformador aprende solo de los datos que recibe en Train.

In [ ]:
df=pd.read_csv(DATA_DIR/'actividad_economica_demo.csv').drop_duplicates()
TARGET='crecimiento_actividad_pct'

num_cols=['inflacion_pct','desempleo_pct','exportaciones_mill','inversion_mill','precio_petroleo_usd','indice_confianza']
cat_cols=['region','sector']

# División simple para demostrar el pipeline.
train,test=train_test_split(df,test_size=.2,random_state=42)

In [ ]:
# Flujo para variables numéricas.
num_pipe=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

# Flujo para variables categóricas.
cat_pipe=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('onehot',OneHotEncoder(handle_unknown='ignore'))
])

# ColumnTransformer aplica cada flujo a sus columnas correspondientes.
pre=ColumnTransformer([
    ('num',num_pipe,num_cols),
    ('cat',cat_pipe,cat_cols)
])

### Qué ocurrió dentro de `pipe.fit()`

1. el imputador numérico aprendió medianas de Train;
2. el scaler aprendió medias/desviaciones de Train;
3. el imputador categórico aprendió cómo rellenar categorías;
4. OneHotEncoder aprendió categorías presentes en Train;
5. Ridge recibió la matriz transformada y aprendió sus coeficientes.

`pipe.predict()` repite las transformaciones aprendidas **sin volver a calcularlas con Test**.

In [ ]:
# Pipeline final: preprocesamiento + modelo.
pipe=Pipeline([
    ('preprocesamiento',pre),
    ('modelo',Ridge(alpha=1.0))
])

pipe.fit(train[num_cols+cat_cols],train[TARGET])
pred=pipe.predict(test[num_cols+cat_cols])

print('MAE :',round(mean_absolute_error(test[TARGET],pred),3))
print('RMSE:',round(np.sqrt(mean_squared_error(test[TARGET],pred)),3))
print('R²  :',round(r2_score(test[TARGET],pred),3))

## 1.1 Inspeccionar qué produjo el preprocesamiento

Un Pipeline no debe ser una caja negra. Podemos recuperar los nombres de las features generadas por `ColumnTransformer`.

Un Pipeline no tiene por qué ser una caja negra. `get_feature_names_out()` muestra las columnas finales después de one-hot. Esto es útil para auditoría, interpretación y depuración.

In [ ]:
# Después de fit, el preprocesador ya conoce categorías y estadísticas.
nombres=pipe.named_steps['preprocesamiento'].get_feature_names_out()
print('Número de features transformadas:',len(nombres))
print(nombres[:20])

## 1.2 Cambiar el modelo sin reescribir la limpieza

Una ventaja del Pipeline es que podemos mantener exactamente el mismo preprocesamiento y sustituir únicamente el estimador final.

Una gran ventaja es desacoplar preprocesamiento y estimador. Podemos sustituir Ridge por RandomForest manteniendo exactamente la misma lógica de preparación, y así comparar de forma más justa.

In [ ]:
pipe_rf=Pipeline([
    ('preprocesamiento',pre),
    ('modelo',RandomForestRegressor(n_estimators=250,max_depth=7,random_state=42,n_jobs=-1))
])
pipe_rf.fit(train[num_cols+cat_cols],train[TARGET])
pred_rf=pipe_rf.predict(test[num_cols+cat_cols])

print('Ridge MAE:',round(mean_absolute_error(test[TARGET],pred),3))
print('RF MAE   :',round(mean_absolute_error(test[TARGET],pred_rf),3))

# 2. Forecasting: por qué una serie temporal es diferente

En forecasting queremos predecir **el futuro**.

Por eso no debemos mezclar aleatoriamente meses futuros dentro de Train.

```text
2014 ... 2022 -> Train
2023          -> Validation
2024-2025     -> Test
```

Además necesitamos transformar historia en features, por ejemplo:

- `lag_1`: valor del mes anterior;
- `lag_3`: hace 3 meses;
- `lag_12`: mismo mes del año anterior;
- media móvil.

En forecasting, el orden temporal cambia las reglas. Queremos predecir **futuro usando solo información disponible en el pasado**.

Un split aleatorio puede permitir que observaciones futuras ayuden indirectamente a entrenar un modelo que luego “predice” el pasado. Eso produciría una evaluación demasiado optimista.

La pregunta correcta siempre incluye un instante de predicción: **¿qué sé exactamente en t cuando quiero predecir t+1?**

In [ ]:
ts=pd.read_csv(DATA_DIR/'macro_mensual_demo.csv',parse_dates=['fecha'])
ts=ts.sort_values('fecha').reset_index(drop=True)

ts.head()

In [ ]:
plt.figure(figsize=(11,4))
plt.plot(ts['fecha'],ts['indice_actividad'])
plt.title('Índice de actividad - serie SINTÉTICA DIDÁCTICA')
plt.xlabel('Fecha')
plt.ylabel('Índice')
plt.show()

## 2.0 Antes de modelar una serie: frecuencia, huecos y orden

Una serie temporal debe revisarse como serie:

- ¿está ordenada?
- ¿hay meses faltantes?
- ¿la frecuencia es mensual/trimestral/anual?
- ¿hubo cambios metodológicos?
- ¿existen eventos extraordinarios?

Revisa:

- frecuencia regular (mensual/trimestral);
- huecos en fechas;
- duplicados por periodo;
- cambios metodológicos;
- tendencia/estacionalidad;
- fecha de publicación real de cada predictor.

Una serie perfectamente ordenada en el CSV no garantiza que sus variables estuvieran disponibles en tiempo real.

In [ ]:
print('Desde:',ts['fecha'].min().date())
print('Hasta:',ts['fecha'].max().date())
print('Filas:',len(ts))
print('Fechas duplicadas:',ts['fecha'].duplicated().sum())

# Diferencia entre fechas consecutivas, útil para detectar huecos.
print(ts['fecha'].sort_values().diff().value_counts().head())

## 2.1 Crear lags

`shift(1)` desplaza la serie una fila: en marzo coloca el valor de febrero. Eso representa información disponible antes de predecir marzo.

Un lag convierte el historial en features supervisadas:

$$lag_1(t)=y_{t-1},\quad lag_{12}(t)=y_{t-12}
$$

Para predecir el mes t, esos valores sí pertenecen al pasado. Después de crear lags aparecen nulos al inicio porque no existe historia suficiente; normalmente se eliminan esas primeras filas.

In [ ]:
serie=ts.copy()

serie['lag_1']=serie['indice_actividad'].shift(1)
serie['lag_3']=serie['indice_actividad'].shift(3)
serie['lag_12']=serie['indice_actividad'].shift(12)

# Media de los 3 meses ANTERIORES. shift(1) evita incluir el propio mes objetivo.
serie['media_3']=serie['indice_actividad'].shift(1).rolling(3).mean()

serie[['fecha','indice_actividad','lag_1','lag_3','lag_12','media_3']].head(15)

In [ ]:
# Las primeras filas no tienen historia suficiente y se eliminan después de crear lags.
serie=serie.dropna().reset_index(drop=True)
print(serie.shape)

## 2.2 Evitar leakage al crear medias móviles

Esta expresión sería peligrosa si incluye el valor del mismo mes objetivo:

```python
rolling(3).mean()
```

Por eso usamos primero `shift(1)` y después `rolling(3)`: en el mes t solo mira t-1, t-2 y t-3.

Error frecuente:

```python
df['media3'] = df['y'].rolling(3).mean()
```

Si se usa esa columna para predecir `y` de la misma fila, la media incluye el valor actual y filtra la respuesta. Primero desplazamos (`shift(1)`) y luego calculamos la ventana para usar **solo pasado**.

In [ ]:
# Verificamos manualmente una fila.
i=20
print('Fecha objetivo:',serie.loc[i,'fecha'])
print('lag_1:',serie.loc[i,'lag_1'])
print('valor del mes anterior:',serie.loc[i-1,'indice_actividad'])

# 3. Split cronológico

Usaremos los primeros 70% de meses para entrenar, siguientes 15% para validación y últimos 15% como Test.

No usamos `shuffle=True`.

La regla es simple: Train contiene periodos anteriores, Validation periodos posteriores y Test los más recientes. No mezclamos fechas. El objetivo es simular cómo se usaría el modelo en el mundo real.

In [ ]:
n=len(serie)
i_train=int(n*.70)
i_val=int(n*.85)

train_ts=serie.iloc[:i_train]
val_ts=serie.iloc[i_train:i_val]
test_ts=serie.iloc[i_val:]

print('Train:',train_ts['fecha'].min().date(),'->',train_ts['fecha'].max().date())
print('Val  :',val_ts['fecha'].min().date(),'->',val_ts['fecha'].max().date())
print('Test :',test_ts['fecha'].min().date(),'->',test_ts['fecha'].max().date())

## 3.1 TimeSeriesSplit para validación cruzada temporal

En series temporales no usamos folds aleatorios. `TimeSeriesSplit` crea ventanas donde Train siempre está antes que Validation.

`TimeSeriesSplit` crea folds expandiendo el pasado y validando en bloques posteriores. A diferencia de K-Fold aleatorio, respeta el orden temporal.

Esto ayuda a saber si el modelo es estable en distintos periodos, aunque sigue siendo necesario reservar un Test final si queremos una evaluación independiente.

In [ ]:

tscv=TimeSeriesSplit(n_splits=4)
for fold,(tr_idx,va_idx) in enumerate(tscv.split(train_ts),start=1):
    print(
        f'Fold {fold}:',
        train_ts.iloc[tr_idx]['fecha'].min().date(),'->',train_ts.iloc[tr_idx]['fecha'].max().date(),
        '| Val:',train_ts.iloc[va_idx]['fecha'].min().date(),'->',train_ts.iloc[va_idx]['fecha'].max().date()
    )

# 4. Baseline temporal

Un baseline muy fuerte en series es **naive**:

> predicción del próximo mes = valor del mes anterior.

Si nuestro modelo sofisticado no supera ese baseline, no está aportando valor predictivo.

Un modelo complejo debe superar referencias simples. Dos baselines habituales:

- **naive:** próximo valor ≈ último valor observado;
- **seasonal naive:** próximo valor ≈ valor del mismo mes del año anterior.

Si Random Forest apenas mejora estos baselines, debemos cuestionar si la complejidad añadida tiene valor.

In [ ]:
pred_naive=val_ts['lag_1']

print('Naive MAE :',round(mean_absolute_error(val_ts['indice_actividad'],pred_naive),3))
print('Naive RMSE:',round(np.sqrt(mean_squared_error(val_ts['indice_actividad'],pred_naive)),3))

## 4.1 Baseline estacional

Para datos mensuales también puede ser fuerte predecir con el mismo mes del año anterior (`lag_12`).

In [ ]:
pred_seasonal=val_ts['lag_12']

baseline_table=pd.DataFrame({
    'Naive_lag1':{
        'MAE':mean_absolute_error(val_ts['indice_actividad'],pred_naive),
        'RMSE':np.sqrt(mean_squared_error(val_ts['indice_actividad'],pred_naive))
    },
    'Seasonal_lag12':{
        'MAE':mean_absolute_error(val_ts['indice_actividad'],pred_seasonal),
        'RMSE':np.sqrt(mean_squared_error(val_ts['indice_actividad'],pred_seasonal))
    }
}).T
baseline_table.round(3)

Un modelo ML debe competir contra baselines razonables, no contra “nada”. En forecasting, superar una predicción naive puede ser más difícil de lo que parece.

# 5. Modelo de forecasting supervisado

Convertimos la serie en una tabla normal:

```text
lag_1, lag_3, lag_12, media_3, inflación, petróleo, exportaciones -> indice_actividad
```

Esto permite usar modelos clásicos de regresión.

Al crear lags y otras variables, convertimos forecasting en una tabla supervisada. El modelo no “sabe” que existe tiempo; somos nosotros quienes preservamos orden y construimos features temporales correctamente.

In [ ]:
features_ts=['lag_1','lag_3','lag_12','media_3','inflacion_interanual_pct','precio_petroleo_usd','exportaciones_mill']
target_ts='indice_actividad'

modelo_ts=RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

modelo_ts.fit(train_ts[features_ts],train_ts[target_ts])
pred_val=modelo_ts.predict(val_ts[features_ts])

print('Modelo MAE :',round(mean_absolute_error(val_ts[target_ts],pred_val),3))
print('Modelo RMSE:',round(np.sqrt(mean_squared_error(val_ts[target_ts],pred_val)),3))

In [ ]:
plt.figure(figsize=(11,4))
plt.plot(val_ts['fecha'],val_ts[target_ts],marker='o',label='Real')
plt.plot(val_ts['fecha'],pred_naive,marker='o',label='Naive lag1')
plt.plot(val_ts['fecha'],pred_val,marker='o',label='Random Forest')
plt.title('Forecasting en Validation')
plt.legend()
plt.show()

## 5.1 Métricas adicionales para series: WAPE y Bias

WAPE facilita expresar error acumulado como porcentaje del volumen real. Bias muestra tendencia sistemática a sobre/subestimar.

Además de MAE/RMSE:

$$WAPE=\frac{\sum |y-\hat y|}{\sum |y|}\times100$$

facilita expresar error agregado como porcentaje de la magnitud real.

$$Bias=\frac{1}{n}\sum(\hat y-y)$$

indica sobre/subestimación sistemática.

Un Bias cercano a cero puede ocultar errores grandes que se cancelan, así que nunca lo miramos solo.

In [ ]:
def metricas_forecast(real,pred):
    real=np.asarray(real); pred=np.asarray(pred)
    return pd.Series({
        'MAE':mean_absolute_error(real,pred),
        'RMSE':np.sqrt(mean_squared_error(real,pred)),
        'WAPE_%':100*np.sum(np.abs(real-pred))/np.sum(np.abs(real)),
        'Bias':np.mean(pred-real)
    })

pd.DataFrame({
    'Naive':metricas_forecast(val_ts[target_ts],pred_naive),
    'RF':metricas_forecast(val_ts[target_ts],pred_val)
}).T.round(3)

## 5.2 Errores a lo largo del tiempo

Un promedio puede ocultar periodos problemáticos. Graficar errores por fecha ayuda a detectar cambios de régimen, shocks o drift.

Una métrica agregada puede ocultar periodos de crisis o cambios estructurales. Graficar residuos por fecha ayuda a detectar drift, estacionalidad residual o periodos donde el modelo falla sistemáticamente.

In [ ]:
errores=val_ts[['fecha']].copy()
errores['error']=val_ts[target_ts].to_numpy()-pred_val

plt.figure(figsize=(10,4))
plt.bar(errores['fecha'],errores['error'])
plt.axhline(0,linestyle='--',linewidth=1)
plt.title('Errores del modelo a través del tiempo')
plt.ylabel('Real - Predicción')
plt.show()

# 6. Cerrar diseño y tocar Test una vez

Solo después de decidir features, modelo e hiperparámetros usando Train/Validation, podemos medir Test.

Si volvemos a modificar el modelo después de mirar Test, debemos reconocer que Test ya dejó de ser completamente independiente.

Test se usa cuando ya cerramos features, modelo e hiperparámetros. Si Test sale peor de lo esperado, **no deberíamos iterar indefinidamente sobre él** y seguir llamándolo Test. En un proyecto real necesitaríamos documentar el hallazgo y, si rediseñamos, conseguir una nueva evaluación futura/holdout.

In [ ]:
# Reentrenamos con Train + Validation porque el diseño ya quedó fijado.
train_final=pd.concat([train_ts,val_ts],ignore_index=True)

modelo_final=RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)
modelo_final.fit(train_final[features_ts],train_final[target_ts])
pred_test=modelo_final.predict(test_ts[features_ts])

print('TEST MAE :',round(mean_absolute_error(test_ts[target_ts],pred_test),3))
print('TEST RMSE:',round(np.sqrt(mean_squared_error(test_ts[target_ts],pred_test)),3))
print('TEST R²  :',round(r2_score(test_ts[target_ts],pred_test),3))

## 6.1 Test no es el final del ciclo de vida

Después del despliegue aparecen problemas nuevos:

- **data drift:** cambian las distribuciones de X;
- **concept drift:** cambia la relación entre X e y;
- nuevas categorías;
- cambios metodológicos de la fuente;
- datos faltantes operativos.

Un modelo productivo necesita monitoreo y revalidación periódica.

Después del despliegue pueden cambiar distribuciones, relaciones y procesos de negocio. Esto se llama **drift**. Un modelo válido hoy necesita monitoreo, revalidación y eventualmente reentrenamiento.

# 7. Cómo presentar resultados

Una presentación técnica breve debería responder:

1. **Problema:** ¿qué decisión queremos apoyar?
2. **Datos:** fuente, período, unidad de observación, tamaño.
3. **Calidad:** nulos, duplicados, outliers, limitaciones.
4. **Split:** cómo se separó Train/Validation/Test.
5. **Baseline:** referencia mínima.
6. **Modelo:** por qué se escogió.
7. **Métricas:** no solo el mejor número, también errores relevantes.
8. **Interpretación:** variables/patrones importantes sin afirmar causalidad injustificada.
9. **Limitaciones:** datos, drift, leakage potencial, sesgos.
10. **Siguiente paso:** cómo validar o desplegar.

Una presentación útil no empieza por “usé Random Forest”. Empieza por la decisión/problema:

1. qué quisimos predecir;
2. qué datos estaban disponibles;
3. qué baseline debía superarse;
4. cómo se validó sin leakage;
5. qué error tiene el modelo en unidades comprensibles;
6. dónde falla;
7. qué limitaciones impiden convertir predicción en afirmación causal o decisión automática.

## 7.1 Una tabla ejecutiva de comparación

La presentación debe permitir comparar modelos sin obligar a leer código.

In [ ]:
# Ejemplo de formato que puedes reutilizar en tu proyecto.
resumen_ejecutivo=pd.DataFrame([
    {'Modelo':'Baseline','MAE':3.2,'RMSE':4.5,'Comentario':'Referencia sencilla'},
    {'Modelo':'Modelo A','MAE':2.6,'RMSE':3.9,'Comentario':'Mejora estable'},
    {'Modelo':'Modelo B','MAE':2.5,'RMSE':5.1,'Comentario':'Algunos errores extremos'}
])
resumen_ejecutivo

El “mejor” modelo no siempre es el que tiene una décima menos de MAE. También importan estabilidad, interpretabilidad, costo de operación, latencia y riesgo.

## Plantilla del proyecto final

### A. Fuente y objetivo
- URL / institución / archivo.
- ¿Qué representa una fila?
- Target o pregunta de clustering.

### B. Exploración
- `shape`, `info`, `describe`.
- nulos, duplicados, categorías, outliers.
- visualizaciones clave.

### C. Preparación
- split apropiado.
- encoding/escalado/imputación.
- baseline.

### D. Modelos
- mínimo 2 alternativas comparables.
- hiperparámetros relevantes.

### E. Evaluación
- métricas justificadas.
- Train vs Validation.
- Test final.

### F. Conclusión
- qué aprendimos;
- qué NO podemos afirmar;
- limitaciones y próximos pasos.

# Checklist final del módulo

- [ ] Definí si es regresión, clasificación o clustering.
- [ ] Entendí filas, columnas y target.
- [ ] Revisé tipos, nulos, duplicados y outliers.
- [ ] Evité leakage.
- [ ] Elegí split apropiado.
- [ ] Construí baseline.
- [ ] Entiendo qué aprende el modelo.
- [ ] Sé qué hiperparámetros estoy modificando.
- [ ] Uso métricas alineadas al problema.
- [ ] Comparo Train y Validation.
- [ ] No uso Test para ajustar repetidamente.
- [ ] No confundo importancia predictiva con causalidad.
- [ ] Puedo explicar el resultado a una persona no técnica.

## Preguntas finales del módulo

- ¿puedes explicar qué representa una fila antes de entrenar?
- ¿puedes distinguir parámetro de hiperparámetro?
- ¿puedes justificar Train/Validation/Test?
- ¿puedes escoger métricas según el costo del error?
- ¿puedes reconocer leakage?
- ¿puedes diferenciar asociación, importancia predictiva y causalidad?
- ¿puedes explicar por qué un modelo más complejo no es automáticamente mejor?
- ¿puedes construir un pipeline reproducible y auditable?
- ¿puedes explicar las limitaciones de tu resultado a alguien no técnico?